# Geo-VLA — train the vision models on a free GPU

Run this on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or Kaggle.
It trains the two execution heads and produces files you upload in the
Geo-VLA **Training studio → Upload checkpoint**, where you can compare them with
earlier versions and promote the best one to production.

| Model | Dataset | Typical time on a T4 |
|---|---|---|
| ResNet-50 land-cover classifier | EuroSAT RGB (27k patches) | ~10 min for 10 epochs |
| Siamese U-Net change detector | LEVIR-CD (10k patches) | ~1 h for 50 epochs |

In [ ]:
!git clone -b claude/new-project-setup-x2fnaz https://github.com/koushik2456/Geo-vla.git
%cd Geo-vla
!pip install -q segmentation-models-pytorch==0.5.0 rasterio geopandas python-dotenv
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to GPU")

## 1 · Land-cover classifier (EuroSAT)
EuroSAT downloads automatically. If the mirror is down, upload `EuroSAT_RGB.zip`, unzip it and add `--image-folder /content/EuroSAT_RGB`.

In [ ]:
!python -m training.train_classifier --epochs 10 --batch-size 128 --workers 2 \
    --out models/checkpoints/resnet50_eurosat.pth

## 2 · Change detector (LEVIR-CD)
Download LEVIR-CD from https://justchenhao.github.io/LEVIR/ (or a mirror such as Kaggle) and arrange it as `data/levir_cd/{train,val,test}/{A,B,label}`.

In [ ]:
import os
assert os.path.isdir("data/levir_cd/train/A"), "put LEVIR-CD into data/levir_cd first (see above)"
!python -m training.train_change_detector --epochs 50 --batch-size 16 --workers 2 \
    --out models/checkpoints/siamese_unet_levircd.pth

## 3 · Download the results
Upload each `.pth` together with its `.metrics.json` in the Geo-VLA Training studio.

In [ ]:
!ls -lh models/checkpoints/
from google.colab import files
for name in ["resnet50_eurosat", "siamese_unet_levircd"]:
    for ext in (".pth", ".metrics.json"):
        path = f"models/checkpoints/{name}{ext}"
        if os.path.exists(path):
            files.download(path)